# Train balance bot using PPO with commands

Run each cell by pressing `shift + enter`.

We repeat our training but with added commands. We spend the first couple of phases getting the bot to balance (without any velocity or turning commands) before adding in the commands. Once the bot handles commands in an ideal environment, we add domain randomization (DR) back in.

Important takeaway: make sure the bot is behaving as intended at the end of each phase before moving on!

In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

# Import custom environment
from balance_bot_env_cmd import BalanceBotEnv, DomainRandomConfig

# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import PPO training module and exporter
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx
from rl.onnx_actor_to_c import export_onnx_actor_to_c

In [2]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 8               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

## Configure PPO and environment

In [3]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 32,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 32,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [4]:
def make_balance_bot_env(render, debug, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path = MJCF_PATH,
        render_mode = "human" if render else None,
        debug = debug,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), debug=(i==0), kw=kwargs: 
                make_balance_bot_env(render, debug=debug, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [5]:
def load_agent(envs, model_path):
    """
    For debugging only! Use this to load a previously trained model to skip previous phases. Note 
    that you will still need to run the cells in each prior phase that update the experiment name 
    and environment.
    """
    from rl.ppo_trainer import Agent, TrainResult

    # Make sure model_path is a Path
    model_path = Path(model_path)
    
    # Load agent from previous run
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = Agent(envs, ppo_config).to(device)
    agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    agent.eval()
    print(f"Loaded model from {model_path}")
    
    # Wrap agent in a dummy result
    return TrainResult(
        agent = agent,
        checkpoint_dir = model_path.parent,
        best_model_path = model_path,
        final_model_path = None,
        best_mean_return = 0,
    )

## Phase 1: Balance only

We'll start with the easy task of only penalizing if the robot tilts (pitches) forward.

In [6]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-01"

# Create an environment that only rewards staying upright
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    vel_reward_coef=0.0,
    yaw_reward_coef=0.0,
)

In [7]:
# Choo choo train
result = train(ppo_config, envs=envs)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-01__42__1780332542
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-01
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.166 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-1.072 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=-0.247 rad, pitch_penalty=0.0306
  pitch_rate=-0.338 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.216 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.433 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=-0.035 rad, pitch_penalty=0.0006
  pitch_rate=-1.731 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.276 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-1.921 rad/s
 

In [8]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.003 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-0.055 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=-0.001 rad, pitch_penalty=0.0000
  pitch_rate=-0.348 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.010 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.235 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=0.017 rad, pitch_penalty=0.0001
  pitch_rate=-0.204 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.020 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-0.063 rad/s
  vel_tracking_reward=0.0000 (perfect=1.0)
  yaw_tracking_reward=0.0000 (perfect=1.0)
  pitch=0.004 rad, pitch_penalty=0.0000
  pitch_rate=-0.

## Phase 2: Small velocity command, penalize pitch rate and rotation (yaw)

In [9]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-02"

# Update domain randomization config
dr = DomainRandomConfig(
    cmd_vel_range=(-0.3, 0.3),  # Slow max forward/back commands
    cmd_yaw_range=(0.0, 0.0),   # No yaw commands yet
    cmd_zero_prob=0.3,          # Ensure bot stays still 30% of the time
)

# Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr
    env.vel_reward_coef = 1.0
    env.yaw_reward_coef = 1.0
    env.sigma_vel = 0.15
    env.sigma_yaw = 0.5
    env.vel_max = 0.5
    env.yaw_max = 1.5
    env.pitch_rate_penalty_coef = 0.05

# Update the PPO config to help prevent getting stuck in a local maximum
ppo_config.ent_coef = 0.01      # Encourage the agent to explore more  
ppo_config.value_clip = 10.0    # Allow for more aggressive critic updates

In [10]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-01__42__1779580717/best_model.cleanrl_model")

In [11]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-02__42__1780334681
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-02
New episode: cmd_vel=-0.037, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=-0.037, vel_target=-0.018 m/s, vel_actual=-0.002 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-0.049 rad/s
  vel_tracking_reward=0.9982 (perfect=1.0)
  yaw_tracking_reward=0.9952 (perfect=1.0)
  pitch=0.002 rad, pitch_penalty=0.0000
  pitch_rate=-0.265 rad/s, pitch_rate_penalty=0.0035
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=-0.037, vel_target=-0.018 m/s, vel_actual=-0.003 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.136 rad/s
  vel_tracking_reward=0.9985 (perfect=1.0)
  yaw_tracking_reward=0.9639 (perfect=1.0)
  pitch=0.007 rad, pitch_penalty=0.0000
  pitch_rate=0.115 rad/s, pitch_rate_penalty=0.0007
  action_smoothness_penalty=0.0000
--- step 3000 ---
  cmd_vel=-0.037, vel_target=-0.018 m/s, vel_actual=0.002 m/s
  cmd_yaw=0.000, yaw

In [12]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.011 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.096 rad/s
  vel_tracking_reward=0.9992 (perfect=1.0)
  yaw_tracking_reward=0.9819 (perfect=1.0)
  pitch=-0.008 rad, pitch_penalty=0.0000
  pitch_rate=0.536 rad/s, pitch_rate_penalty=0.0144
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.005 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.109 rad/s
  vel_tracking_reward=0.9998 (perfect=1.0)
  yaw_tracking_reward=0.9766 (perfect=1.0)
  pitch=-0.012 rad, pitch_penalty=0.0001
  pitch_rate=-0.110 rad/s, pitch_rate_penalty=0.0006
  action_smoothness_penalty=0.0000
New episode: cmd_vel=-0.050, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=-0.050, vel_target=-0.025 m/s, vel_actual=0.001 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.136 rad/s
  vel_tracking_reward=0.9955 (perfect=1.0)
  yaw_tracking

## Phase 3: Tighten velocity tracking

In [13]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-03"

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.sigma_vel = 0.005     # Small sigma: less reward for error

In [14]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-02__42__1779582576/best_model.cleanrl_model")

In [15]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-03__42__1780337846
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-03
New episode: cmd_vel=-0.037, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=-0.037, vel_target=-0.018 m/s, vel_actual=-0.001 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=-0.061 rad/s
  vel_tracking_reward=0.9437 (perfect=1.0)
  yaw_tracking_reward=0.9927 (perfect=1.0)
  pitch=-0.013 rad, pitch_penalty=0.0001
  pitch_rate=-0.028 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=-0.037, vel_target=-0.018 m/s, vel_actual=0.012 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.087 rad/s
  vel_tracking_reward=0.8368 (perfect=1.0)
  yaw_tracking_reward=0.9849 (perfect=1.0)
  pitch=-0.001 rad, pitch_penalty=0.0000
  pitch_rate=0.170 rad/s, pitch_rate_penalty=0.0014
  action_smoothness_penalty=0.0000
--- step 3000 ---
  cmd_vel=-0.037, vel_target=-0.018 m/s, vel_actual=-0.004 m/s
  cmd_yaw=0.000, y

In [16]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory,
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.001 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.001 rad/s
  vel_tracking_reward=0.9998 (perfect=1.0)
  yaw_tracking_reward=1.0000 (perfect=1.0)
  pitch=0.014 rad, pitch_penalty=0.0001
  pitch_rate=0.067 rad/s, pitch_rate_penalty=0.0002
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.001 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.013 rad/s
  vel_tracking_reward=0.9998 (perfect=1.0)
  yaw_tracking_reward=0.9996 (perfect=1.0)
  pitch=0.013 rad, pitch_penalty=0.0001
  pitch_rate=-0.053 rad/s, pitch_rate_penalty=0.0001
  action_smoothness_penalty=0.0000
New episode: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.001 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.014 rad/s
  vel_tracking_reward=0.9999 (perfect=1.0)
  yaw_tracking_rewa

## Phase 4: Introduce small yaw commands

In [17]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-04"

# Update domain randomization config
dr.cmd_vel_range = (-0.3, 0.3)
dr.cmd_yaw_range = (-0.5, 0.5)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr
    env.sigma_yaw = 0.15        # Tighten to punish deviating from yaw command
    env.yaw_max = 1.5           # Slow max turn (rad/s)

In [18]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-04__42__1780339508
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-04
New episode: cmd_vel=-0.037, cmd_yaw=0.359
--- step 1000 ---
  cmd_vel=-0.037, vel_target=-0.018 m/s, vel_actual=-0.024 m/s
  cmd_yaw=0.359, yaw_target=0.538 rad/s, yaw_actual=-0.035 rad/s
  vel_tracking_reward=0.9940 (perfect=1.0)
  yaw_tracking_reward=0.1125 (perfect=1.0)
  pitch=-0.026 rad, pitch_penalty=0.0003
  pitch_rate=0.008 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0000
--- step 2000 ---
  cmd_vel=-0.037, vel_target=-0.018 m/s, vel_actual=-0.021 m/s
  cmd_yaw=0.359, yaw_target=0.538 rad/s, yaw_actual=0.001 rad/s
  vel_tracking_reward=0.9986 (perfect=1.0)
  yaw_tracking_reward=0.1460 (perfect=1.0)
  pitch=-0.025 rad, pitch_penalty=0.0003
  pitch_rate=0.049 rad/s, pitch_rate_penalty=0.0001
  action_smoothness_penalty=0.0000
--- step 3000 ---
  cmd_vel=-0.037, vel_target=-0.018 m/s, vel_actual=-0.027 m/s
  cmd_yaw=0.359, y

In [19]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.295, cmd_yaw=0.392
New episode: cmd_vel=0.236, cmd_yaw=0.019
Mean return: 655.52
Exported charts_metrics.csv (5 metrics, 449 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 5: Mid-episode command sampling and smoothness penalty

Keep same vel/yaw command range. Any larger, and it seems the bot will tip over. Add small chance of generating a new set of commands every few episodes.

In [20]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-05"

# Update domain randomization config
dr.cmd_vel_range = (-0.3, 0.3)
dr.cmd_yaw_range = (-0.5, 0.5)
dr.cmd_resample_prob = 0.005    # 0.5% chance of new command every step

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr
    env.smoothness_penalty_coef = 0.1

In [21]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-04__42__1779748283/best_model.cleanrl_model")

In [22]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-05__42__1780340991
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-05
New episode: cmd_vel=-0.037, cmd_yaw=0.359
New command: cmd_vel=0.106, cmd_yaw=-0.470
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.006, cmd_yaw=-0.167
New command: cmd_vel=0.071, cmd_yaw=-0.337
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.131, cmd_yaw=-0.438
--- step 1000 ---
  cmd_vel=-0.131, vel_target=-0.066 m/s, vel_actual=-0.023 m/s
  cmd_yaw=-0.438, yaw_target=-0.657 rad/s, yaw_actual=-0.639 rad/s
  vel_tracking_reward=0.6904 (perfect=1.0)
  yaw_tracking_reward=0.9979 (perfect=1.0)
  pitch=-0.018 rad, pitch_penalty=0.0002
  pitch_rate=-0.120 rad/s, pitch_rate_penalty=0.0007
  action_smoothness_penalty=0.0006
New command: cmd_vel=0.165, cmd_yaw=-0.465
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.296, cmd_yaw=-0.461
New command: cmd_vel=0.162,

In [23]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=-0.294, cmd_yaw=0.412
New command: cmd_vel=0.126, cmd_yaw=0.225
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.253, cmd_yaw=-0.360
New command: cmd_vel=0.008, cmd_yaw=-0.288
New command: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=-0.001 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.012 rad/s
  vel_tracking_reward=0.9999 (perfect=1.0)
  yaw_tracking_reward=0.9990 (perfect=1.0)
  pitch=0.003 rad, pitch_penalty=0.0000
  pitch_rate=0.131 rad/s, pitch_rate_penalty=0.0009
  action_smoothness_penalty=0.0039
New command: cmd_vel=-0.002, cmd_yaw=0.331
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.282, cmd_yaw=-0.398
New command: cmd_vel=0.021, cmd_yaw=0.214
New command: cmd_vel=0.049, cmd_yaw=0.279
New command: cmd_vel=0.174, cmd_yaw=-0.033
New command: cmd_vel=-0.088, cmd_yaw=0.126
New command: cmd_vel=0.285, cmd_yaw=0.197
New command: cmd_vel=-0.015, cmd_yaw=0.243
--- s

## Phase 6: Observation noise and action delay

In [24]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-06"

# Update domain randomization config
dr.pitch_noise_std_dev=0.01         # Inject some noise into pitch observation
dr.pitch_rate_noise_std_dev=0.01    # Inject some noise into pitch rate observation
dr.wheel_vel_noise_std_dev = 0.1    # Inject some noise into wheel velocity observation
dr.action_delay_steps=1             # 1 step (5ms) delay
dr.action_delay_random=True,        # Vary 0-1 steps each episode

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr

In [25]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-06__42__1780342939
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-06
New episode: cmd_vel=-0.037, cmd_yaw=0.359
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.269, cmd_yaw=0.257
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.131, cmd_yaw=0.067
New command: cmd_vel=-0.288, cmd_yaw=-0.081
New command: cmd_vel=-0.117, cmd_yaw=-0.320
New command: cmd_vel=-0.150, cmd_yaw=0.246
--- step 1000 ---
  cmd_vel=-0.150, vel_target=-0.075 m/s, vel_actual=-0.067 m/s
  cmd_yaw=0.246, yaw_target=0.369 rad/s, yaw_actual=0.322 rad/s
  vel_tracking_reward=0.9863 (perfect=1.0)
  yaw_tracking_reward=0.9855 (perfect=1.0)
  pitch=-0.011 rad, pitch_penalty=0.0003
  pitch_rate=-0.168 rad/s, pitch_rate_penalty=0.0014
  action_smoothness_penalty=0.0013
New command: cmd_vel=-0.292, cmd_yaw=0.410
New command: cmd_vel=-0.256, cmd

In [26]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.109, cmd_yaw=0.348
New command: cmd_vel=-0.072, cmd_yaw=-0.124
New command: cmd_vel=-0.042, cmd_yaw=-0.391
New command: cmd_vel=0.000, cmd_yaw=0.000
--- step 1000 ---
  cmd_vel=0.000, vel_target=0.000 m/s, vel_actual=0.003 m/s
  cmd_yaw=0.000, yaw_target=0.000 rad/s, yaw_actual=0.015 rad/s
  vel_tracking_reward=0.9981 (perfect=1.0)
  yaw_tracking_reward=0.9985 (perfect=1.0)
  pitch=0.004 rad, pitch_penalty=0.0000
  pitch_rate=0.003 rad/s, pitch_rate_penalty=0.0000
  action_smoothness_penalty=0.0026
New command: cmd_vel=-0.069, cmd_yaw=-0.275
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.229, cmd_yaw=-0.377
New command: cmd_vel=-0.139, cmd_yaw=-0.310
New command: cmd_vel=0.088, cmd_yaw=-0.141
--- step 2000 ---
  cmd_vel=0.088, vel_target=0.044 m/s, vel_actual=0.014 m/s
  cmd_yaw=-0.141, yaw_target=-0.212 rad/s, yaw_actual=-0.061 rad/s
  vel_tracking_reward=0.837

## Phase 7: Add motor noise and random pushes

In [27]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-07"

# Update domain randomization config
dr.motor_noise_scale = 0.02 # +/-2% motor noise
dr.push_prob = 0.005        # 0.5% chance of push per step
dr.push_force_max_n = 0.3   # Gentle pushes

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr

In [28]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-07__42__1780345280
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-07
New episode: cmd_vel=-0.037, cmd_yaw=0.359
New command: cmd_vel=0.296, cmd_yaw=-0.461
New command: cmd_vel=0.066, cmd_yaw=-0.371
New command: cmd_vel=0.131, cmd_yaw=0.067
New command: cmd_vel=-0.288, cmd_yaw=-0.081
New command: cmd_vel=-0.150, cmd_yaw=0.246
New command: cmd_vel=0.054, cmd_yaw=-0.045
New command: cmd_vel=0.114, cmd_yaw=0.062
New command: cmd_vel=0.270, cmd_yaw=0.073
New command: cmd_vel=-0.122, cmd_yaw=0.057
--- step 1000 ---
  cmd_vel=-0.122, vel_target=-0.061 m/s, vel_actual=0.101 m/s
  cmd_yaw=0.057, yaw_target=0.086 rad/s, yaw_actual=0.090 rad/s
  vel_tracking_reward=0.0052 (perfect=1.0)
  yaw_tracking_reward=0.9999 (perfect=1.0)
  pitch=-0.112 rad, pitch_penalty=0.0064
  pitch_rate=-1.560 rad/s, pitch_rate_penalty=0.1217
  action_smoothness_penalty=0.0035
New command: cmd_vel=0.158, cmd_yaw=-0.082
New command: cmd_vel=0.196, cmd

In [29]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.075, cmd_yaw=-0.105
New command: cmd_vel=-0.109, cmd_yaw=-0.055
--- step 1000 ---
  cmd_vel=-0.109, vel_target=-0.055 m/s, vel_actual=-0.047 m/s
  cmd_yaw=-0.055, yaw_target=-0.082 rad/s, yaw_actual=-0.008 rad/s
  vel_tracking_reward=0.9875 (perfect=1.0)
  yaw_tracking_reward=0.9641 (perfect=1.0)
  pitch=-0.033 rad, pitch_penalty=0.0001
  pitch_rate=0.486 rad/s, pitch_rate_penalty=0.0118
  action_smoothness_penalty=0.0045
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.071, cmd_yaw=0.304
--- step 2000 ---
  cmd_vel=0.071, vel_target=0.036 m/s, vel_actual=0.026 m/s
  cmd_yaw=0.304, yaw_target=0.456 rad/s, yaw_actual=0.517 rad/s
  vel_tracking_reward=0.9828 (perfect=1.0)
  yaw_tracking_reward=0.9753 (perfect=1.0)
  pitch=0.011 rad, pitch_penalty=0.0001
  pitch_rate=-0.253 rad/s, pitch_rate_penalty=0.0032
  action_smoothness_penalty=0.0024
New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.088, cmd_yaw=-0.205
New command: cmd_vel=0.078

## Phase 8: Add mass and friction randomization

In [30]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-08"

# Update domain randomization config
dr.mass_scale_range= (0.8, 1.2)       # +/-20% mass variation
dr.friction_scale_range = (0.5, 1.5)  # 50-150% friction variation

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr

In [31]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-08__42__1780347324
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-08
New episode: cmd_vel=0.118, cmd_yaw=-0.406
New command: cmd_vel=0.106, cmd_yaw=-0.470
New command: cmd_vel=0.296, cmd_yaw=-0.461
New command: cmd_vel=0.066, cmd_yaw=-0.371
New command: cmd_vel=0.131, cmd_yaw=0.067
New command: cmd_vel=-0.288, cmd_yaw=-0.081
New command: cmd_vel=-0.150, cmd_yaw=0.246
New command: cmd_vel=0.054, cmd_yaw=-0.045
New command: cmd_vel=0.114, cmd_yaw=0.062
New command: cmd_vel=0.270, cmd_yaw=0.073
New command: cmd_vel=-0.122, cmd_yaw=0.057
--- step 1000 ---
  cmd_vel=-0.122, vel_target=-0.061 m/s, vel_actual=0.087 m/s
  cmd_yaw=0.057, yaw_target=0.086 rad/s, yaw_actual=0.233 rad/s
  vel_tracking_reward=0.0127 (perfect=1.0)
  yaw_tracking_reward=0.8661 (perfect=1.0)
  pitch=-0.181 rad, pitch_penalty=0.0139
  pitch_rate=-1.869 rad/s, pitch_rate_penalty=0.1747
  action_smoothness_penalty=0.0009
New command: cmd_vel=0.158, cmd

In [32]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.109, cmd_yaw=-0.055
--- step 1000 ---
  cmd_vel=-0.109, vel_target=-0.055 m/s, vel_actual=-0.039 m/s
  cmd_yaw=-0.055, yaw_target=-0.082 rad/s, yaw_actual=-0.171 rad/s
  vel_tracking_reward=0.9530 (perfect=1.0)
  yaw_tracking_reward=0.9487 (perfect=1.0)
  pitch=-0.047 rad, pitch_penalty=0.0014
  pitch_rate=0.154 rad/s, pitch_rate_penalty=0.0012
  action_smoothness_penalty=0.0026
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.071, cmd_yaw=0.304
--- step 2000 ---
  cmd_vel=0.071, vel_target=0.036 m/s, vel_actual=0.037 m/s
  cmd_yaw=0.304, yaw_target=0.456 rad/s, yaw_actual=0.530 rad/s
  vel_tracking_reward=0.9998 (perfect=1.0)
  yaw_tracking_reward=0.9638 (perfect=1.0)
  pitch=0.012 rad, pitch_penalty=0.0000
  pitch_rate=-0.242 rad/s, pitch_rate_penalty=0.0029
  action_smoothness_penalty=0.0001
New episode: cmd_vel=0.039, cmd_yaw=0.121
New command: cmd_vel=0.078, cmd_yaw=-0.180
New command: cmd_vel=0.025,

## Phase 9: Add motor gain randomization

In [33]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-09"

# Update domain randomization config
dr.motor_gain_range = (0.6, 1.0)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr
    env.smoothness_penalty_coef = 0.5

In [34]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-08__42__1779826517/best_model.cleanrl_model")

In [35]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-09__42__1780349318
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-09
New episode: cmd_vel=-0.243, cmd_yaw=0.476
New command: cmd_vel=0.106, cmd_yaw=-0.470
New command: cmd_vel=0.296, cmd_yaw=-0.461
New command: cmd_vel=0.066, cmd_yaw=-0.371
New command: cmd_vel=0.131, cmd_yaw=0.067
New command: cmd_vel=-0.288, cmd_yaw=-0.081
New command: cmd_vel=-0.150, cmd_yaw=0.246
New command: cmd_vel=0.054, cmd_yaw=-0.045
New command: cmd_vel=0.114, cmd_yaw=0.062
New command: cmd_vel=0.270, cmd_yaw=0.073
New command: cmd_vel=-0.122, cmd_yaw=0.057
--- step 1000 ---
  cmd_vel=-0.122, vel_target=-0.061 m/s, vel_actual=0.098 m/s
  cmd_yaw=0.057, yaw_target=0.086 rad/s, yaw_actual=0.175 rad/s
  vel_tracking_reward=0.0063 (perfect=1.0)
  yaw_tracking_reward=0.9482 (perfect=1.0)
  pitch=-0.204 rad, pitch_penalty=0.0179
  pitch_rate=-1.978 rad/s, pitch_rate_penalty=0.1956
  action_smoothness_penalty=0.0033
New command: cmd_vel=0.158, cmd

In [36]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.243, cmd_yaw=0.307
--- step 1000 ---
  cmd_vel=0.243, vel_target=0.122 m/s, vel_actual=0.104 m/s
  cmd_yaw=0.307, yaw_target=0.461 rad/s, yaw_actual=0.412 rad/s
  vel_tracking_reward=0.9417 (perfect=1.0)
  yaw_tracking_reward=0.9844 (perfect=1.0)
  pitch=0.024 rad, pitch_penalty=0.0002
  pitch_rate=0.099 rad/s, pitch_rate_penalty=0.0005
  action_smoothness_penalty=0.0203
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.071, cmd_yaw=0.304
--- step 2000 ---
  cmd_vel=0.071, vel_target=0.036 m/s, vel_actual=0.028 m/s
  cmd_yaw=0.304, yaw_target=0.456 rad/s, yaw_actual=0.533 rad/s
  vel_tracking_reward=0.9893 (perfect=1.0)
  yaw_tracking_reward=0.9616 (perfect=1.0)
  pitch=0.016 rad, pitch_penalty=0.0001
  pitch_rate=-0.182 rad/s, pitch_rate_penalty=0.0017
  action_smoothness_penalty=0.0007
New episode: cmd_vel=0.072, cmd_yaw=-0.267
New command: cmd_vel=0.088, cmd_yaw=-0.205
New command: cmd_vel=0.078, cmd_yaw=-0.180
New command: cmd_vel=0.025, cmd_ya

## Phase 10: Random axle torques to simulate ridges

In [37]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-10"

# Update domain randomization config
dr.ridge_prob=0.05              # 5% possibility of tire "ridge"
dr.ridge_torque_max_nm=0.005    # Add a slight torque to the joints (simulate tire ridges)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.dr = dr

In [38]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-09__42__1779828587/best_model.cleanrl_model")

In [39]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-10__42__1780351285
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-10
New episode: cmd_vel=-0.243, cmd_yaw=0.476
New command: cmd_vel=0.106, cmd_yaw=-0.470
New command: cmd_vel=-0.131, cmd_yaw=-0.438
New command: cmd_vel=0.165, cmd_yaw=-0.465
New command: cmd_vel=0.269, cmd_yaw=0.257
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.150, cmd_yaw=0.246
New command: cmd_vel=0.218, cmd_yaw=0.014
New command: cmd_vel=0.270, cmd_yaw=0.073
New command: cmd_vel=-0.074, cmd_yaw=0.478
--- step 1000 ---
  cmd_vel=-0.074, vel_target=-0.037 m/s, vel_actual=-0.019 m/s
  cmd_yaw=0.478, yaw_target=0.717 rad/s, yaw_actual=0.823 rad/s
  vel_tracking_reward=0.9395 (perfect=1.0)
  yaw_tracking_reward=0.9278 (perfect=1.0)
  pitch=-0.017 rad, pitch_penalty=0.0004
  pitch_rate=-0.410 rad/s, pitch_rate_penalty=0.0084
  action_smoothness_penalty=0.0271
New command: cmd_vel=-0.074, cmd_yaw=0.292
New command: cmd_vel=0.158, cmd

In [40]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3,
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

New episode: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=0.213, cmd_yaw=-0.393
New command: cmd_vel=0.032, cmd_yaw=-0.383
New command: cmd_vel=0.116, cmd_yaw=-0.453
--- step 1000 ---
  cmd_vel=0.116, vel_target=0.058 m/s, vel_actual=0.071 m/s
  cmd_yaw=-0.453, yaw_target=-0.680 rad/s, yaw_actual=-0.602 rad/s
  vel_tracking_reward=0.9682 (perfect=1.0)
  yaw_tracking_reward=0.9602 (perfect=1.0)
  pitch=0.024 rad, pitch_penalty=0.0001
  pitch_rate=0.302 rad/s, pitch_rate_penalty=0.0046
  action_smoothness_penalty=0.0505
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.287, cmd_yaw=0.158
New command: cmd_vel=-0.234, cmd_yaw=-0.018
New command: cmd_vel=0.141, cmd_yaw=0.406
New command: cmd_vel=-0.261, cmd_yaw=0.425
New command: cmd_vel=-0.153, cmd_yaw=0.076
New command: cmd_vel=0.000, cmd_yaw=0.000
New command: cmd_vel=-0.066, cmd_yaw=-0.315
--- step 2000 ---
  cmd_vel=-0.066, vel_target=-0.033 m/s, vel_actual=0.000 m/s
  cmd_yaw=-0.315, yaw_target=-0.473 rad/s, yaw_a

## Clean up and save model

At this point, we are done with training. We want to delete the environments and save the best actor from our final training phase. We'll export this actor network as an ONNX file that can be used on a variety of hardware platforms.

In [41]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3
Closing env 4
Closing env 5
Closing env 6
Closing env 7


In [42]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

/workspace/software/rl/ppo_trainer.py:381: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0601 23:06:57.543000 155146 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0601 23:06:57.545000 155146 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0601 23:06:57.549000 155146 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs/BalanceBot-v0__balance-bot-phase-10__42__1780351285/actor.onnx


/opt/pyenv/versions/3.12.13/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [43]:
# Export actor to .h file
export_onnx_actor_to_c(
    onnx_path   = result.checkpoint_dir / "actor.onnx",
    output_path = result.checkpoint_dir / "actor.h"
)

Weights found:
  0.weight: shape=(32, 6), dtype=float32
  0.bias: shape=(32,), dtype=float32
  2.weight: shape=(32, 32), dtype=float32
  2.bias: shape=(32,), dtype=float32
  4.weight: shape=(2, 32), dtype=float32
  4.bias: shape=(2,), dtype=float32
C header written to runs/BalanceBot-v0__balance-bot-phase-10__42__1780351285/actor.h
  Layers:  3
  Obs:     6
  Actions: 2


PosixPath('runs/BalanceBot-v0__balance-bot-phase-10__42__1780351285/actor.h')